In [3]:
# ============================================================================
# BLOCK 5: TOPIC MODELING - ROMANIAN
# Input: 04_sentiment_data_ro.pkl
# Output: 05_topics_data_ro.pkl
# Runtime: ~15-20 minutes on CPU
# ============================================================================
%run 00_setup_and_config.ipynb

✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 10:45:24
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [4]:
# ============================================================================
# BLOCK 5: TOPIC MODELING - ROMANIAN
# ============================================================================

print('='*70)
print('BLOCK 5: TOPIC MODELING - ROMANIAN')
print('='*70)

# Delete old checkpoints to force re-run
import os, shutil
CHECKPOINT_FILE = PROCESSED_DIR / '05_topics_data_ro.pkl'
MODEL_FILE = PROCESSED_DIR / 'bertopic_model_ro'

if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    print('⚠️  Deleted old checkpoint - will re-run topic modeling')

if MODEL_FILE.exists():
    shutil.rmtree(MODEL_FILE)
    print('⚠️  Deleted old model - will re-run topic modeling')

# Load from PREVIOUS checkpoint (04_sentiment_data)
df_ro = load_checkpoint('ro', '04_sentiment_data')
if df_ro is None:
    raise FileNotFoundError('Run 03_sentiment_analysis_ro.ipynb first')

print(f'\nRunning BERTopic on {len(df_ro):,} Romanian comments...')

# Load embedding model
print('\nLoading embedding model...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model = embedding_model.to('cpu')
print('✓ Embedding model loaded')

# ============================================================================
# IMPORTS
# ============================================================================
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
import gc

gc.collect()

# ============================================================================
# 📍 CREATE VECTORIZER HERE (BEFORE BERTOPIC)
# ============================================================================
print('\nCreating CountVectorizer with Romanian stopwords...')

vectorizer_ro = CountVectorizer(
    stop_words=STOPWORDS_RO,      # ← From 00_setup_and_config.ipynb
    ngram_range=(2, 4),           # Only phrases (no single words)
    min_df=2,                    # Word must appear in 10+ documents
    max_df=0.95,                   # Word can't appear in >70% of documents
    lowercase=True,
    token_pattern=r'(?u)\b\w[\w-]*\w\b|\w+\b'
)

print(f'✓ CountVectorizer created with {len(STOPWORDS_RO)} stopwords')

# ============================================================================
# CREATE BERTOPIC MODEL (PASS VECTORIZER HERE)
# ============================================================================
topic_model = BERTopic(
    language='multilingual',
    embedding_model=embedding_model,
    min_topic_size=15,
    nr_topics=8,
    calculate_probabilities=True,
    verbose=True,
    vectorizer_model=vectorizer_ro,  # ← PASS VECTORIZER HERE
    representation_model=KeyBERTInspired()
)

print('✓ BERTopic model initialized')

# ============================================================================
# PREPARE TEXTS
# ============================================================================
print('\nPreparing texts...')

texts = df_ro['clean_text'].fillna('').astype(str).tolist()

# Filter very short texts (less than 3 words)
texts_filtered = []
indices_kept = []
for i, t in enumerate(texts):
    if len(t.split()) >= 3:
        texts_filtered.append(t)
        indices_kept.append(i)

print(f'  Original texts: {len(texts):,}')
print(f'  After filtering: {len(texts_filtered):,}')
print(f'  Removed: {len(texts) - len(texts_filtered):,} short texts')

# ============================================================================
# FIT MODEL
# ============================================================================
print('\nFitting BERTopic model...')
topics, probs = topic_model.fit_transform(texts_filtered)

# Map back to original dataframe
df_ro['topic'] = -1
df_ro['topic_probability'] = 0.0

for idx, (topic, prob) in zip(indices_kept, zip(topics, probs)):
    df_ro.loc[idx, 'topic'] = topic
    df_ro.loc[idx, 'topic_probability'] = max(prob)

# Save model
topic_model.save(PROCESSED_DIR / 'bertopic_model_ro')
print(f'✓ Topic model saved')

# Save checkpoint
save_checkpoint(df_ro, 'ro', '05_topics_data')
update_pipeline_status('ro', 5, 'completed')

# ============================================================================
# QUALITY CHECK
# ============================================================================
print('\n' + '='*70)
print('TOPIC QUALITY REPORT')
print('='*70)

topic_info = topic_model.get_topic_info()
display(topic_info)

# Count outliers
outliers = (df_ro['topic'] == -1).sum()
outlier_pct = 100 * outliers / len(df_ro)
print(f'\n⚠️  OUTLIERS (Topic -1): {outliers:,} comments ({outlier_pct:.1f}%)')

if outlier_pct > 30:
    print('   ⚠️  WARNING: High outlier rate - consider reducing min_topic_size')
else:
    print('   ✓ Outlier rate acceptable')

# ============================================================================
# SHOW TOPIC WORDS (FIXED ITERATION)
# ============================================================================
print('\n--- TOP 5 WORDS PER TOPIC ---')

for _, row in topic_info.iterrows():
    topic_id = int(row['Topic'])  # Ensure it's an integer
    
    # Skip outliers
    if topic_id == -1:
        continue
    
    # Get topic words
    topic_words = topic_model.get_topic(topic_id)
    
    # Safety check
    if not topic_words or not isinstance(topic_words, list):
        print(f"⚠️  Topic {topic_id}: Could not retrieve words")
        continue
    
    words = [w for w, s in topic_words[:5]]
    count = int(row['Count'])
    print(f"Topic {topic_id} ({count:,} comments): {', '.join(words)}")


# ============================================================================
# VISUALIZATION
# ============================================================================
print('\n' + '='*70)
print('TOPIC SUMMARY - ROMANIAN')
print('='*70)

topic_info.to_csv(OUTPUT_DIR / 'romanian' / 'ro_topics_summary.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'romanian' / 'ro_topics_summary.csv'}")

print('\nGenerating topic visualization...')

fig, ax = plt.subplots(figsize=(14, 8))
topic_sizes = topic_model.get_topic_info().sort_values('Count', ascending=False)
colors = plt.cm.Set3(np.linspace(0, 1, len(topic_sizes)))

ax.barh(range(len(topic_sizes)), topic_sizes['Count'].values, color=colors)
ax.set_yticks(range(len(topic_sizes)))
ax.set_yticklabels([f"Topic {i}: {rep[:50]}..." 
                    for i, rep in zip(topic_sizes['Topic'], topic_sizes['Representation'])])
ax.set_xlabel('Number of Comments', fontsize=12)
ax.set_title(f'Topic Distribution - Romanian (n={len(df_ro):,})', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
save_figure(fig, 'ro_topics_overview.png', 'ro', 'visualizations')

print('\n' + '='*70)
print('✓ BLOCK 5 COMPLETE - TOPIC MODELING DONE')
print('='*70)


BLOCK 5: TOPIC MODELING - ROMANIAN
✓ Loading checkpoint: 04_sentiment_data_ro.pkl

Running BERTopic on 6,600 Romanian comments...

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4047.24it/s]
2026-05-29 10:45:32,764 - BERTopic - Embedding - Transforming documents to embeddings.


✓ Embedding model loaded

Creating CountVectorizer with Romanian stopwords...
✓ CountVectorizer created with 412 stopwords
✓ BERTopic model initialized

Preparing texts...
  Original texts: 6,600
  After filtering: 6,050
  Removed: 550 short texts

Fitting BERTopic model...


Batches: 100%|██████████| 190/190 [02:17<00:00,  1.39it/s]
2026-05-29 10:47:49,912 - BERTopic - Embedding - Completed ✓
2026-05-29 10:47:49,912 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-29 10:48:20,997 - BERTopic - Dimensionality - Completed ✓
2026-05-29 10:48:20,999 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-29 10:48:22,189 - BERTopic - Cluster - Completed ✓
2026-05-29 10:48:22,189 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-29 10:48:22,804 - BERTopic - Representation - Completed ✓
2026-05-29 10:48:22,804 - BERTopic - Topic reduction - Reducing number of topics
2026-05-29 10:48:22,827 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-29 10:48:27,666 - BERTopic - Representation - Completed ✓
2026-05-29 10:48:27,671 - BERTopic - Topic reduction - Reduced number of topics from 62 to 8
2026-05-29 10:48:29,776 - BERTopic - WARNING: When 

✓ Topic model saved
✓ Checkpoint saved: 05_topics_data_ro.pkl
✓ Pipeline status updated: ro - Block 5 - completed

TOPIC QUALITY REPORT


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2959,-1_votez nicușor dan_votați nicușor dan_votat nicușor_votat simion,"[votez nicușor dan, votați nicușor dan, votat nicușor, votat simion, votez nicușor, vota simion,...","[3 voturi din Italia pentru Nicusor Dan, Drumul Românilor e Vestul , EUROPA UNITĂ UE ! NICUȘOR D..."
1,0,1208,0_domnul nicusor dan_dan nicușor_votam nicușor dan_votez nicușor dan,"[domnul nicusor dan, dan nicușor, votam nicușor dan, votez nicușor dan, nicușor dan președinte, ...","[Adică Nicușor Dan ar fi mai bun? Haida de,să nu exageram!, Integritatea și Hărnicie lui Nicușor..."
2,1,803,1_simion presedintele_simion presedinte_calin georgescu presedinte_george simion presedintele,"[simion presedintele, simion presedinte, calin georgescu presedinte, george simion presedintele,...","[Nicusor Dan Presedinte, "" Daca noi castigam, castiga si ei, daca ei castiga, pierdem toti"". Cu ..."
3,2,603,2_simion președintele româniei_george simion președintele româniei_președinte al româniei_președ...,"[simion președintele româniei, george simion președintele româniei, președinte al româniei, preș...","[Simion președintele României, Simion președintele României, George Simion președintele României]"
4,3,357,3_recorder chiar_recorder ați_cei recorder_echipa recorder,"[recorder chiar, recorder ați, cei recorder, echipa recorder, recorder va, astia recorder, multu...","[Se vede de la 10 miliarde de km a cui portavoce Recorder e. #Biased, Recorder e cu USR ce aștep..."
5,4,47,4_iar psd_conducerea psd_guvernare psd_psd psd,"[iar psd, conducerea psd, guvernare psd, psd psd, psd pnl udmr, guvernare psd pnl, psd pnl usr u...",[Să te lupți să nu ajungă PSD în turul 2 și după alegeri să continui cu PSD PNL UDMR asta înseam...
6,5,43,5_videoul asta_ajunga cat multa_ajunga informatii_ajunga cat,"[videoul asta, ajunga cat multa, ajunga informatii, ajunga cat, ajunga cat multa lume, cat multa...","[Comentariu pentru algoritm , sa ajunga la cat mai multa lume., comentariu pt algoritm, sa ajung..."
7,6,30,6_comunitatea lgbt_votez george simion_votez george_votanților simion,"[comunitatea lgbt, votez george simion, votez george, votanților simion, merg votez, lumea votea...",[Problema nu cred ca e daca e gay sau nu. Problema e ca daca chiar e in acelasi timp el este imp...



⚠️  OUTLIERS (Topic -1): 3,679 comments (54.3%)
   ⚠️  WARNING: High outlier rate - consider reducing min_topic_size

--- TOP 5 WORDS PER TOPIC ---
Topic 0 (1,208 comments): domnul nicusor dan, dan nicușor, votam nicușor dan, votez nicușor dan, nicușor dan președinte
Topic 1 (803 comments): simion presedintele, simion presedinte, calin georgescu presedinte, george simion presedintele, calin georgescu
Topic 2 (603 comments): simion președintele româniei, george simion președintele româniei, președinte al româniei, președintele româniei, al româniei
Topic 3 (357 comments): recorder chiar, recorder ați, cei recorder, echipa recorder, recorder va
Topic 4 (47 comments): iar psd, conducerea psd, guvernare psd, psd psd, psd pnl udmr
Topic 5 (43 comments): videoul asta, ajunga cat multa, ajunga informatii, ajunga cat, ajunga cat multa lume
Topic 6 (30 comments): comunitatea lgbt, votez george simion, votez george, votanților simion, merg votez

TOPIC SUMMARY - ROMANIAN

✓ Saved: outputs\roman